# Advanced Charts
**Emotion Heatmap | Radar Chart (Era 1 vs Era 4) | PCA Clusters**

Reads from `data/feature/movies_features.csv` and `data/feature/movies_normalized.csv`.

In [1]:
import sys, os
from math import pi
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

PROJECT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT, "src", "algorithms"))
from config import FEAT_DATA_PATH, NORM_DATA_PATH, OUTPUT_CHART_DIR, F1_COLS, EMOTION_NAMES, ERAS

df = pd.read_csv(FEAT_DATA_PATH)
df_norm = pd.read_csv(NORM_DATA_PATH)
print(f"Loaded {len(df)} films")

Loaded 1466 films


## Chart 1 — Top 15 Emotion Correlation Heatmap

In [2]:
top15_cols = df[F1_COLS].mean().sort_values(ascending=False).head(15).index
corr = df[top15_cols].corr()
labels = [c.replace("f1_", "") for c in top15_cols]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title("Emotion Correlation Matrix (Top 15 Most Common Emotions)", fontsize=15)
plt.tight_layout()
out = os.path.join(OUTPUT_CHART_DIR, "adv_emotion_heatmap.png")
fig.savefig(out, dpi=150)
plt.close(fig)
print(f"Saved: {out}")

Saved: C:\Users\siucomp\Desktop\Review 2\output\charts\adv_emotion_heatmap.png


## Chart 2 — Radar Chart: Era 1 vs Era 4 Emotional DNA

In [3]:
era1_key = [k for k in ERAS if "1920" in k][0]
era4_key = [k for k in ERAS if "2001" in k][0]
era_means = df.groupby("era")[F1_COLS].mean()
top8_cols = df[F1_COLS].mean().sort_values(ascending=False).head(8).index
categories = [c.replace("f1_", "") for c in top8_cols]
N = len(categories)

if era1_key in era_means.index and era4_key in era_means.index:
    v1 = era_means.loc[era1_key, top8_cols].values.flatten().tolist()
    v4 = era_means.loc[era4_key, top8_cols].values.flatten().tolist()
    v1 += v1[:1]
    v4 += v4[:1]
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.plot(angles, v1, linewidth=2, label="Era 1 (Classic 1920-1950)")
    ax.fill(angles, v1, alpha=0.25)
    ax.plot(angles, v4, linewidth=2, label="Era 4 (Modern 2001-2024)")
    ax.fill(angles, v4, alpha=0.25)
    plt.xticks(angles[:-1], categories, size=11)
    ax.set_title("Emotional DNA: Classic vs Modern Cinema", size=15, y=1.1)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    out = os.path.join(OUTPUT_CHART_DIR, "adv_radar_chart.png")
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Saved: {out}")
else:
    print("Era data not found. Check movies_features.csv era column values.")
    print("Available eras:", list(era_means.index))

Saved: C:\Users\siucomp\Desktop\Review 2\output\charts\adv_radar_chart.png


## Chart 3 — PCA Cluster Scatter (from normalized emotion data)

In [4]:
X = df_norm[F1_COLS].fillna(0).values
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

var1 = round(pca.explained_variance_ratio_[0] * 100, 1)
var2 = round(pca.explained_variance_ratio_[1] * 100, 1)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap="viridis", alpha=0.6, s=20)
ax.set_xlabel(f"PC1 ({var1}% variance)", fontsize=12)
ax.set_ylabel(f"PC2 ({var2}% variance)", fontsize=12)
ax.set_title("Film Clusters Based on 50 Emotion Features (KMeans + PCA)", fontsize=14)
handles, _ = scatter.legend_elements()
ax.legend(handles, ["Cluster 1", "Cluster 2", "Cluster 3"], title="Clusters")
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(OUTPUT_CHART_DIR, "adv_pca_clusters.png")
fig.savefig(out, dpi=150)
plt.close(fig)
print(f"Saved: {out}")
print(f"Total variance explained: {var1 + var2}%")

Saved: C:\Users\siucomp\Desktop\Review 2\output\charts\adv_pca_clusters.png
Total variance explained: 22.9%
